<a href="https://colab.research.google.com/github/nazndev/visibility-gated-clip-reid/blob/main/notebooks/colab_iteration2_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Iteration 2, corrected notebook

This file does not replace `notebooks/colab_iteration2.ipynb`. The older notebook keeps the T4 outputs already recorded.

Set `REVISION` to the 40-character commit that contains the audit fixes. The notebook checks out that commit and will not follow a newer `main` by itself. If this commit is not on GitHub yet, Colab cannot run it: push the commit first, paste its hash, then open this notebook again.

Run Cell 1 after every disconnect, then one experiment cell. Do not start a second experiment while one is running.

In [3]:
# Cell 1 — clone the recorded revision. Run this again after a disconnect.
import os
import re
import subprocess
from pathlib import Path

REVISION = "d143dbdbd690f70315cb706403980eddf09279e9"  # paste the 40-character audit-fix commit; do not leave this blank
REPO = "https://github.com/nazndev/visibility-gated-clip-reid.git"
ROOT = Path("/content/lvlm-person-reid")

if not re.fullmatch(r"[0-9a-f]{40}", REVISION):
    raise SystemExit("Set REVISION to the pushed audit-fix commit. This notebook does not track origin/main.")
if not (ROOT / ".git").is_dir():
    subprocess.check_call(["git", "clone", REPO, str(ROOT)])
subprocess.check_call(["git", "fetch", "origin"], cwd=ROOT)
subprocess.check_call(["git", "checkout", "--force", REVISION], cwd=ROOT)
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != REVISION:
    raise SystemExit(f"HEAD {head} is not the recorded revision {REVISION}")
os.chdir(ROOT)
print(head)


d143dbdbd690f70315cb706403980eddf09279e9


In [4]:
# Cell 2 — GPU check.
import os
print("nvidia device", os.path.exists("/dev/nvidia0"))
if not os.path.exists("/dev/nvidia0"):
    raise RuntimeError("Select a GPU runtime, then run this cell again.")


nvidia device True


In [5]:
# Cell 3 — install.
import subprocess
from pathlib import Path
ROOT = Path("/content/lvlm-person-reid")
subprocess.check_call(["pip", "install", "-q", "-r", "requirements-colab.txt"], cwd=ROOT)
subprocess.check_call(["pip", "install", "-q", "-e", "."], cwd=ROOT)
import torch
print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "no cuda")


2.15.0+cu132 True Tesla T4


In [6]:
# Cell 4 — Drive, dataset, and checkpoint.
import subprocess
from pathlib import Path
from google.colab import drive
ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
drive.mount("/content/drive")
DRIVE.mkdir(parents=True, exist_ok=True)
if not (ROOT / "data/Market-1501-v15.09.15/bounding_box_train").is_dir():
    subprocess.check_call(["python", "scripts/01_download_dataset.py"], cwd=ROOT)
if not (ROOT / "checkpoints/Market1501_clipreid_ViT-B-16_60.pth").is_file():
    subprocess.check_call(["python", "scripts/02_download_model.py"], cwd=ROOT)
print(DRIVE)


Mounted at /content/drive
/content/drive/MyDrive/visibility-gated-reid/iteration2_v2


In [ ]:
# Cell 5 — one experiment. Change only this cell, then run it alone.
# Pending on the same T4, in this order:
#   validation / none / keep 128
#   validation / cls_attention / block 6 / 64
#   validation / cls_attention / block 9 / 96
#   validation / random / block 9 / 96 / seeds 11, 23, 37, 53, 71
# The old block-6 random files stay in iteration2/. This cell writes iteration2_v2/.
SPLIT = "validation"
STRATEGY = "none"  # none, cls_attention, random
BLOCK = "6"
KEEP = "128"
SEED = ""  # 11, 23, 37, 53, or 71 when STRATEGY is random
BACKEND = "reference"

import json
import subprocess
from pathlib import Path
ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
if STRATEGY == "random" and SEED not in {"11", "23", "37", "53", "71"}:
    raise SystemExit("Random selection needs SEED 11, 23, 37, 53, or 71.")
name = f"{SPLIT}_{STRATEGY}_b{BLOCK}_k{KEEP}_{BACKEND}" + (f"_s{SEED}" if STRATEGY == "random" else "")
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", SPLIT,
    "--strategy", STRATEGY,
    "--prune-block", BLOCK,
    "--keep", KEEP,
    "--backend", BACKEND,
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
if STRATEGY == "random":
    command.extend(["--seed", SEED])
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


In [7]:
# Cell 5b — validation, class attention, block 6, keep 64. Run this cell alone.
# Cell 5 already finished the full encoder. Do not run Cell 5 again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_cls_attention_b6_k64_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "cls_attention",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy cls_attention --prune-block 6 --keep 64 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b6_k64_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b6_k64_reference.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "1c2cd1835fffe4ff59ca81cf7cd6b69e0abe828df799984612e015d6ad02aa90",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version"

In [ ]:
# Cell 6 — list completed, missing, and failed score files. This does not start a run.
import json
from pathlib import Path
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
expected = [
    "validation_none_b6_k128_reference.json",
    "validation_cls_attention_b6_k64_reference.json",
    "validation_cls_attention_b9_k96_reference.json",
] + [f"validation_random_b9_k96_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
expected += [
    "test_none_b6_k128_reference.json",
    "test_cls_attention_b6_k64_reference.json",
    "test_cls_attention_b9_k96_reference.json",
]
expected += [f"test_random_b6_k64_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
expected += [f"test_random_b9_k96_reference_s{seed}.json" for seed in (11, 23, 37, 53, 71)]
rows = []
for name in expected:
    path = DRIVE / name
    if not path.is_file():
        rows.append({"file": name, "state": "pending"})
        continue
    payload = json.loads(path.read_text())
    scores = payload.get("scores", {})
    rows.append({
        "file": name,
        "state": payload.get("label"),
        "mAP": scores.get("mAP"),
        "rank1": scores.get("rank1"),
        "rank5": scores.get("rank5"),
        "rank10": scores.get("rank10"),
        "valid_queries": scores.get("valid_queries"),
        "excluded_queries": scores.get("excluded_queries"),
        "kreciprocal": payload.get("kreciprocal"),
        "execution_sha256": payload.get("execution_sha256"),
        "git": payload.get("git"),
        "counts": payload.get("counts"),
    })
print(json.dumps(rows, indent=2))


In [8]:
# Cell 5c — validation, class attention, block 9, keep 96. Run this cell alone.
# Cells 5 and 5b are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_cls_attention_b9_k96_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "cls_attention",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy cls_attention --prune-block 9 --keep 96 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b9_k96_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_cls_attention_b9_k96_reference.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "17a5f47ba42af5567ddf4e4b17666ffccd92d4a5a34c0a9a86463ecd52eaadd4",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version"

In [9]:
# Cell 5d — validation, random, block 9, keep 96, seed 11. Run this cell alone.
# Cells 5, 5b, and 5c are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s11"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "11",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 11 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s11 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s11.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "8dbc8b2a62d537d07f0eb3a27f982271b6f67272a44275e007a222b309dcd41b",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [10]:
# Cell 5e — validation, random, block 9, keep 96, seed 23. Run this cell alone.
# Cells 5, 5b, 5c, and 5d are already finished. Do not run them again.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s23"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "23",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 23 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s23 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s23.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "ac82a571c68ce4ec21c16605bbdec6a2edb1160b0cd8b6e1b2c8c6301e6447fb",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [11]:
# Cell 5f — validation, random, block 9, keep 96, seed 37. Run this cell alone.
# Cells 5 through 5e are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s37"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "37",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 37 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s37 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s37.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "26e8ee92cf62706fe2c19ece99ced11f8df9b002d49666f0f40498b286a041b3",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [12]:
# Cell 5g — validation, random, block 9, keep 96, seed 53. Run this cell alone.
# Cells 5 through 5f are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s53"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "53",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 53 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s53 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s53.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "e760af746c515bf491a8cccf186c8490be2d9ebb9208927254b17c63c92a23c4",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [13]:
# Cell 5h — validation, random, block 9, keep 96, seed 71. Run this cell alone.
# Cells 5 through 5g are already finished. Do not run them again, and do not start the next seed until this cell prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "validation_random_b9_k96_reference_s71"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "validation",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "71",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split validation --strategy random --prune-block 9 --keep 96 --backend reference --seed 71 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s71 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/validation_random_b9_k96_reference_s71.json
{
  "evaluation": {
    "protocol": "training-derived validation",
    "identity_seen": true,
    "note": "The encoder was trained on these identities and source training images. This is not an official Market-1501 test result."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "0d6e02ffd808883e65e321b160d34713f2fa706e15575253a314f6bb19aa1c50",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "

In [14]:
# Cell 8 — official Market-1501 test, full encoder. Run this cell alone.
# The validation cells are finished. Do not run them again. Do not start another official-test cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_none_b6_k128_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "none",
    "--prune-block", "6",
    "--keep", "128",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy none --prune-block 6 --keep 128 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_none_b6_k128_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_none_b6_k128_reference.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "05a63c7194f556bb98acf725f58387b16ba47da832642a8ae6a29eb50af6268f",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
    "strategy": "none",


In [15]:
# Cell 8b — official test, class attention, block 6, keep 64. Run this cell alone.
# Cell 8 is already finished. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_cls_attention_b6_k64_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "cls_attention",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy cls_attention --prune-block 6 --keep 64 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_cls_attention_b6_k64_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_cls_attention_b6_k64_reference.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "2b512157deee6f16117e9a666573cf5d746fe90b1b079ef9f630b4231b7a7562",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",


In [16]:
# Cell 8c — official test, random, block 6, keep 64, seed 11. Run this cell alone.
# The earlier official-test cells are already finished or must be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b6_k64_reference_s11"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--seed", "11",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 6 --keep 64 --backend reference --seed 11 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s11 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s11.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "e663af82e6b1c6d08b09d6578f62b80060660883c9e7e633e236f17233ff4fe4",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [17]:
# Cell 8d — official test, random, block 6, keep 64, seed 23. Run this cell alone.
# The earlier official-test cells are already finished or must be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b6_k64_reference_s23"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--seed", "23",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 6 --keep 64 --backend reference --seed 23 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s23 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s23.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "8e4f9dd756798ece033106ca65b67aa29725f88548526a2c4d3d014b6514ab28",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [18]:
# Cell 8e — official test, random, block 6, keep 64, seed 37. Run this cell alone.
# The earlier official-test cells are already finished or must be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b6_k64_reference_s37"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--seed", "37",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 6 --keep 64 --backend reference --seed 37 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s37 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s37.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "4420e4b9df24dc540685d8112a610e736dc5c09fc937e66ffcc06da95acb649e",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [19]:
# Cell 8f — official test, random, block 6, keep 64, seed 53. Run this cell alone.
# The earlier official-test cells are already finished or must be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b6_k64_reference_s53"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--seed", "53",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 6 --keep 64 --backend reference --seed 53 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s53 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s53.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "7971bf0aec59061a6644c8c6e4977dcb1721104fa34d3029401324a655663137",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [20]:
# Cell 8g — official test, random, block 6, keep 64, seed 71. Run this cell alone.
# The earlier official-test cells are already finished or must be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b6_k64_reference_s71"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "6",
    "--keep", "64",
    "--backend", "reference",
    "--seed", "71",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 6 --keep 64 --backend reference --seed 71 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s71 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b6_k64_reference_s71.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "d0729d325d3fa31b917d6469901e3a44c799019ca54c956fa62f03f51e44f37f",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [21]:
# Cell 8h — official test, class attention, block 9, keep 96. Run this cell alone.
# The block-6 official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_cls_attention_b9_k96_reference"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "cls_attention",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy cls_attention --prune-block 9 --keep 96 --backend reference --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_cls_attention_b9_k96_reference --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_cls_attention_b9_k96_reference.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "a781a831af7ff32a2dc3558ba28e9508d580cad80577397cb14438a33ff2724a",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",


In [22]:
# Cell 8i — official test, random, block 9, keep 96, seed 11. Run this cell alone.
# The earlier official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b9_k96_reference_s11"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "11",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 9 --keep 96 --backend reference --seed 11 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s11 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s11.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "eb26e9d63c4a13fc49eec1db0c29bf52d510a1065b104103fd877a625d177269",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [23]:
# Cell 8j — official test, random, block 9, keep 96, seed 23. Run this cell alone.
# The earlier official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b9_k96_reference_s23"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "23",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 9 --keep 96 --backend reference --seed 23 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s23 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s23.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "a9ec9c0f92f860890cf490b953f933adcceadf4dfa3d0a3f377036cd8a26271a",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [24]:
# Cell 8k — official test, random, block 9, keep 96, seed 37. Run this cell alone.
# The earlier official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b9_k96_reference_s37"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "37",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 9 --keep 96 --backend reference --seed 37 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s37 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s37.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "de52c38bb525639c4ad2d18a7042debf1360671fda65d64f67c745f26f20cc24",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [25]:
# Cell 8l — official test, random, block 9, keep 96, seed 53. Run this cell alone.
# The earlier official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b9_k96_reference_s53"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "53",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 9 --keep 96 --backend reference --seed 53 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s53 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s53.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "4555b4af2462811ec5d6a978d7309673e4c50d3459e5ba0d3563166ab63c304c",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   

In [26]:
# Cell 8m — official test, random, block 9, keep 96, seed 71. Run this cell alone.
# The earlier official-test cells should be finished first. Do not run finished cells again, and do not start the next cell until this one prints its scores.
import json
import subprocess
from pathlib import Path

ROOT = Path("/content/lvlm-person-reid")
DRIVE = Path("/content/drive/MyDrive/visibility-gated-reid/iteration2_v2")
if not (ROOT / "src" / "reidvlm").is_dir():
    raise SystemExit("Run Cell 1 first.")
head = subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=ROOT, text=True).strip()
if head != "d143dbdbd690f70315cb706403980eddf09279e9":
    raise SystemExit(f"HEAD is {head}. Run Cell 1 with that revision before this cell.")
name = "test_random_b9_k96_reference_s71"
command = [
    "python", "-m", "reidvlm", "prune-embed",
    "--config", "configs/market1501.yaml",
    "--split", "test",
    "--strategy", "random",
    "--prune-block", "9",
    "--keep", "96",
    "--backend", "reference",
    "--seed", "71",
    "--batch-size", "32",
    "--run-dir", str(DRIVE / name),
    "--output", str(DRIVE / f"{name}.json"),
]
print(" ".join(command))
subprocess.check_call(command, cwd=ROOT)
saved = json.loads((DRIVE / f"{name}.json").read_text())
identity = saved["identity"]
print(json.dumps({
    "evaluation": saved.get("evaluation"),
    "git": saved.get("git"),
    "execution_sha256": identity.get("execution_sha256"),
    "execution_source_sha256": identity.get("execution_source_sha256"),
    "checkpoint_sha256": identity.get("checkpoint_sha256"),
    "selection": identity.get("selection"),
    "counts": saved.get("counts"),
    "valid_queries": saved.get("scores", {}).get("valid_queries"),
    "excluded_queries": saved.get("scores", {}).get("excluded_queries"),
    "mAP": saved.get("scores", {}).get("mAP"),
    "rank1": saved.get("scores", {}).get("rank1"),
    "rank5": saved.get("scores", {}).get("rank5"),
    "rank10": saved.get("scores", {}).get("rank10"),
    "kreciprocal": saved.get("kreciprocal"),
    "limited": identity.get("limited"),
}, indent=2))


python -m reidvlm prune-embed --config configs/market1501.yaml --split test --strategy random --prune-block 9 --keep 96 --backend reference --seed 71 --batch-size 32 --run-dir /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s71 --output /content/drive/MyDrive/visibility-gated-reid/iteration2_v2/test_random_b9_k96_reference_s71.json
{
  "evaluation": {
    "protocol": "official Market-1501 test",
    "identity_seen": false,
    "note": "Official query and gallery lists after junk removal. Same-person same-camera gallery images are excluded at scoring time."
  },
  "git": "d143dbdbd690f70315cb706403980eddf09279e9",
  "execution_sha256": "7c57039be38a08e04eb06823d0f855053242eee4ce9aab570f1159d5b19a3be6",
  "execution_source_sha256": "284a151018860cb7bb6ecad600c4d20ad604a58c47bd3694e9d6ab4af9487210",
  "checkpoint_sha256": "bedc8fcc37f296045df19d901b162dd032bfa85d9b4f2406eff8313536d444dc",
  "selection": {
    "selection_version": "selection-v1",
   